# 2. Embedding, and the cache that makes it survivable

*Part of the [broadlisten](https://github.com/InteractionDataLab/broad-listening/tree/main/code)
stage-by-stage examples. Every number and figure below was produced by running
this notebook on the synthetic corpus in `broadlisten.sample`, which is
generated from templates and contains no participant data.*

Embedding is the only expensive stage that cannot be skipped, so it is the one
that most needs a cache keyed by what actually affects it.

**This notebook does not download a model.** `sample_embeddings` returns
synthetic vectors with the same shape and geometry a real run produces, so
everything downstream is the real code path. The call a real run makes is
`embed_corpus(texts, cfg, store)`, shown at the end.

In [1]:
from broadlisten.config import EMBED_REGISTRY

print(f"{'backend':14s} {'model':32s} dims")
for name, (model, dim) in EMBED_REGISTRY.items():
    print(f"{name:14s} {model:32s} {dim}")

backend        model                            dims
qwen3-8b       Qwen/Qwen3-Embedding-8B          4096
qwen3          Qwen/Qwen3-Embedding-4B          2560
qwen3-0.6b     Qwen/Qwen3-Embedding-0.6B        1024
bge-m3         BAAI/bge-m3                      1024


## One immutable settings object

`Config` is frozen. `with_` returns a new one rather than mutating this one, so
a cache key can never drift out from under a stage that already ran.

In [2]:
from broadlisten.config import Config

cfg = Config.create("responses.csv", text_cols=["answer"])
print(f"embed model : {cfg.embed_model} ({cfg.embed_dim} dims)")
print(f"seed        : {cfg.seed}")
print(f"batch       : {cfg.embed_batch}")

embed model : Qwen/Qwen3-Embedding-8B (4096 dims)
seed        : 42
batch       : 16


## The point of the cache key

Each stage's key is built only from the settings that stage depends on. Change
`k_max` and the clustering re-runs; the embedding, which may be hours, does
not.

In [3]:
louder = cfg.with_(k_max=40)

print("changing k_max ...")
print(f"  embed key   same? {cfg.fingerprint('embed', 600) == louder.fingerprint('embed', 600)}")
print(f"  cluster key same? {cfg.fingerprint('cluster', 600) == louder.fingerprint('cluster', 600)}")

smaller = cfg.with_(embed_backend="qwen3-0.6b")
print("\nchanging the embedding model ...")
print(f"  embed key   same? {cfg.fingerprint('embed', 600) == smaller.fingerprint('embed', 600)}")

changing k_max ...
  embed key   same? True
  cluster key same? False

changing the embedding model ...
  embed key   same? False


## The vectors

Synthetic here, real in a real run. The shape and the geometry are what the
later stages care about.

In [4]:
import numpy as np
from broadlisten.sample import sample_corpus, sample_embeddings

corpus = sample_corpus(seed=0)
vectors = sample_embeddings(corpus.planted_theme.tolist(), seed=0)
print(f"vectors: {vectors.shape}, dtype {vectors.dtype}")
print(f"norms:   mean {np.linalg.norm(vectors, axis=1).mean():.3f}")

vectors: (600, 64), dtype float32
norms:   mean 1.000


## What a real run calls instead

```python
from broadlisten.embed import embed_corpus
from broadlisten.cache import ArtifactStore

cfg = Config.create("responses.csv", text_cols=["answer"],
                    embed_backend="qwen3-8b")
cfg.apply_environment()      # must run before torch is imported
store = ArtifactStore.from_config(cfg)
vectors = embed_corpus(texts, cfg, store)   # cached on the text itself
```

Nothing is sent anywhere: the model is downloaded once and runs locally.

**Next:** [3. Reduction](03_reduce.ipynb), and how much it distorts.